# MovieLens latest-small 探索的データ分析

## 目的

MovieLens latest-smallの評価データと映画データを確認し、映画推薦システムの設計に必要な特徴や問題点を把握する。

## このNotebookで確認する内容

- ユーザー数、映画数、評価数
- 評価値の分布
- ユーザーごとの評価件数
- 映画ごとの評価件数
- 評価件数が少ない映画の割合
- ジャンルごとの映画数
- 評価データの疎らさ

> 現在は最初の作業単位として、データの読み込みと構造確認までを実装する。

## 1. ライブラリの読み込み

- `pathlib.Path`：ファイルやフォルダのパスを扱う
- `pandas`：CSVをDataFrameとして読み込み、表形式データを操作する
- `matplotlib.pyplot`：グラフを作成する（後のEDAで使用）

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

## 2. データファイルの場所を確認する

Notebookを`notebooks`フォルダから実行した場合と、プロジェクト直下から実行した場合の両方に対応して、プロジェクトの基準位置を決める。

`Path.exists()`は、指定したファイルが存在すれば`True`、存在しなければ`False`を返す。

In [2]:
# Notebookの現在の実行位置を確認する
current_dir = Path.cwd()

# notebooksフォルダで実行している場合は、1つ上をプロジェクトの基準位置にする
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

data_dir = project_root / "data" / "raw" / "ml-latest-small"
ratings_path = data_dir / "ratings.csv"
movies_path = data_dir / "movies.csv"

print("現在の実行位置:", current_dir)
print("データフォルダ:", data_dir)
print("ratings.csvが存在するか:", ratings_path.exists())
print("movies.csvが存在するか:", movies_path.exists())

現在の実行位置: /Users/ma-ya/Free/movie-recommendation-system/notebooks
データフォルダ: /Users/ma-ya/Free/movie-recommendation-system/data/raw/ml-latest-small
ratings.csvが存在するか: True
movies.csvが存在するか: True


## 3. CSVをDataFrameとして読み込む

`pd.read_csv()`はCSVファイルのパスを入力として受け取り、表形式の`DataFrame`を返す。

データがまだ配置されていない場合は、期待する配置先を示すエラーで処理を止める。

In [3]:
# 必要なCSVが両方そろっていることを読み込み前に確認する
if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        "MovieLensのCSVが見つかりません。"
        f"{data_dir} にratings.csvとmovies.csvを配置してください。"
    )

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print("CSVの読み込みが完了しました。")

CSVの読み込みが完了しました。


## 4. 読み込み結果を確認する

- `DataFrame.shape`：`(行数, 列数)`を返す
- `DataFrame.head()`：先頭5行をDataFrameとして返す
- `display()`：DataFrameをNotebook上で表として表示する

In [4]:
print("ratingsの形:", ratings.shape)
print("moviesの形:", movies.shape)

print("ratingsの先頭5行")
display(ratings.head())

print("moviesの先頭5行")
display(movies.head())

ratingsの形: (100836, 4)
moviesの形: (9742, 3)
ratingsの先頭5行


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


moviesの先頭5行


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## この時点で確認すること

- `ratings.csv`と`movies.csv`がどちらも見つかるか
- `ratings`がおよそ100,836行・4列か
- `movies`がおよそ9,742行・3列か
- `userId`、`movieId`、`rating`、`timestamp`の意味を説明できるか
- `movieId`、`title`、`genres`の意味を説明できるか

次の作業単位では、欠損値、重複、データ型、評価値の範囲を確認する。

## データ構造の確認結果

- `ratings.csv`と`movies.csv`が存在し、想定した行数・列数で読み込めた。現時点では「ファイルが破損していない」と断定するのではなく、「想定した形式で読み込めた」と表現する。
- `userId`は、評価を行ったユーザーを識別する匿名IDである。
- `movieId`は映画を識別するIDであり、`ratings`と`movies`を結合する共通キーである。
- `rating`はユーザーが映画に付けた評価値である。先頭行では4.0や5.0を確認できたが、データ全体が想定範囲内かは、最小値・最大値・固有値を集計して確認する必要がある。
- `timestamp`は評価された日時を表し、過去の評価を学習用、未来の評価を評価用にする時系列分割で利用できる。
- `title`は映画名であり、主に推薦結果を人が理解できる形で表示するために利用する。映画の識別には`movieId`を使用する。
- `genres`は映画の大まかな特徴を表す。今回は主推薦方式には直接使用しないが、EDA、推薦結果の解釈、改善案の検討に利用できる。

### 現時点では未確認の内容

行数と列数が想定どおりでも、各セルの欠損値、重複、範囲外の評価値、`ratings`にしか存在しない`movieId`が含まれている可能性は残る。これらは次の分析で個別に確認する。

## 5. データ品質の確認

推薦システムの計算に使用する前に、必要な列、データ型、欠損値を確認する。この段階ではデータを修正せず、問題の有無を確認する。

### 5.1 必要な列の確認

必要な列名と実際の列名を集合として比較する。入力は2つの列名集合、出力は不足している列名の集合である。結果が空集合の`set()`なら不足はない。

In [5]:
# 推薦と評価に必要な列を定義する
required_ratings_columns = {"userId", "movieId", "rating", "timestamp"}
required_movies_columns = {"movieId", "title", "genres"}

# 必要な列のうち、実際のデータに存在しない列を調べる
missing_ratings_columns = required_ratings_columns - set(ratings.columns)
missing_movies_columns = required_movies_columns - set(movies.columns)

print("ratingsで不足している列:", missing_ratings_columns)
print("moviesで不足している列:", missing_movies_columns)

ratingsで不足している列: set()
moviesで不足している列: set()


### 5.2 データ型の確認

`DataFrame.dtypes`は、各列のデータ型をSeriesとして返す。ID、評価値、時刻が数値計算に使える型で読み込まれているか確認する。

In [6]:
print("ratingsのデータ型")
display(ratings.dtypes)

print("moviesのデータ型")
display(movies.dtypes)

ratingsのデータ型


userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

moviesのデータ型


movieId     int64
title      object
genres     object
dtype: object

### 5.3 欠損値の確認

`DataFrame.isna()`は各セルが欠損なら`True`を返し、`sum()`は列ごとの`True`を合計する。入力はDataFrame、出力は列ごとの欠損値数を持つSeriesである。未評価のユーザーと映画の組み合わせは、ここでいう欠損値とは異なる。

In [7]:
# 列ごとの欠損値数を集計する
ratings_missing = ratings.isna().sum()
movies_missing = movies.isna().sum()

print("ratingsの欠損値数")
display(ratings_missing)

print("moviesの欠損値数")
display(movies_missing)

ratingsの欠損値数


userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

moviesの欠損値数


movieId    0
title      0
genres     0
dtype: int64

### 5.1〜5.3の確認結果

- `ratings`と`movies`に不足している列はなかった。
- `userId`、`movieId`、`timestamp`は整数型、`rating`は小数型として読み込まれていた。
- `title`と`genres`は文字列を扱う`object`型として読み込まれていた。
- `ratings`と`movies`のすべての列で、pandasが欠損値として認識するセルは0件だった。
- 必要な列、データ型、欠損値には問題が見つからなかったため、次に重複と評価値の妥当性を確認する。

### 5.4 重複の確認

重複を確認する基本形は`DataFrame.duplicated(subset=None, keep="first")`である。

- `subset`：重複判定に使う列を指定する。省略すると全列を使う
- `keep="first"`：最初の行を残し、2行目以降を`True`にする（初期値）
- `keep=False`：重複グループに含まれるすべての行を`True`にする
- 出力：各行が重複かを示す真偽値のSeries

`duplicated().sum()`で2行目以降の重複件数を数え、`keep=False`で該当グループ全体を抽出できる。今回は次の3種類を分けて確認する。

1. `ratings`の全列が同じ完全重複
2. `ratings`で`userId`と`movieId`が同じ組み合わせ
3. `movies`の完全重複と`movieId`の重複

同じユーザー・映画の評価が複数あると、評価行列の1つのマスに複数値が対応してしまう。`movies.movieId`が重複していると、結合時に評価行が増える可能性がある。

In [8]:
# ratingsで全列が同じ完全重複を数える
ratings_exact_duplicate_count = int(ratings.duplicated().sum())

# 同じユーザーと映画の組み合わせに含まれる全行を抽出する
user_movie_duplicate_mask = ratings.duplicated(
    subset=["userId", "movieId"],
    keep=False,
)
user_movie_duplicate_rows = ratings.loc[user_movie_duplicate_mask].sort_values(
    ["userId", "movieId", "timestamp"]
)
user_movie_duplicate_pair_count = (
    user_movie_duplicate_rows[["userId", "movieId"]]
    .drop_duplicates()
    .shape[0]
)

# moviesの完全重複と、識別キーmovieIdの重複を数える
movies_exact_duplicate_count = int(movies.duplicated().sum())
movie_id_duplicate_count = int(
    movies.duplicated(subset=["movieId"]).sum()
)

print("ratingsの完全重複行数:", ratings_exact_duplicate_count)
print("同じuserId・movieIdを持つ行数:", len(user_movie_duplicate_rows))
print("重複しているuserId・movieIdの組数:", user_movie_duplicate_pair_count)
print("moviesの完全重複行数:", movies_exact_duplicate_count)
print("moviesで重複しているmovieId数:", movie_id_duplicate_count)

# 重複が見つかった場合だけ、先頭10行を表示して内容を確認する
if not user_movie_duplicate_rows.empty:
    display(user_movie_duplicate_rows.head(10))

ratingsの完全重複行数: 0
同じuserId・movieIdを持つ行数: 0
重複しているuserId・movieIdの組数: 0
moviesの完全重複行数: 0
moviesで重複しているmovieId数: 0


#### 重複確認コードの読み方

- `ratings.loc[マスク]`：マスクが`True`の行だけを返す
- `sort_values([...])`：指定した列の順に行を並べる
- `drop_duplicates()`：同じ値を持つ重複行を1行にまとめる
- `shape[0]`：DataFrameの行数を返す
- `DataFrame.empty`：行が0件なら`True`を返す

重複が見つかった場合はすぐ削除せず、評価値と`timestamp`を見て原因と扱いを判断する。

### 5.5 評価値の妥当性

MovieLens latest-smallの`rating`は0.5〜5.0の0.5刻みを想定する。最小値と最大値だけでは途中の不正値を発見できないため、観測された評価値の種類と許可値に含まれない件数も確認する。

- `Series.min()`：列の最小値を返す
- `Series.max()`：列の最大値を返す
- `Series.unique()`：重複を除いた値をNumPy配列として返す
- `NumPy配列.tolist()`：NumPy配列を通常のPythonリストへ変換する
- `sorted()`：受け取った値を昇順のリストとして返す
- `Series.isin(値の集合)`：各値が集合に含まれるかを真偽値で返す
- `~`：`True`と`False`を反転し、許可値に含まれない行を抽出する

In [9]:
# MovieLensで許可される評価値を明示する
valid_rating_values = {
    0.5, 1.0, 1.5, 2.0, 2.5,
    3.0, 3.5, 4.0, 4.5, 5.0,
}

rating_min = ratings["rating"].min()
rating_max = ratings["rating"].max()
observed_rating_values = sorted(ratings["rating"].unique().tolist())

# 許可値に含まれない評価を抽出するマスクを作る
invalid_rating_mask = ~ratings["rating"].isin(valid_rating_values)
invalid_ratings = ratings.loc[invalid_rating_mask]

print("評価値の最小値:", rating_min)
print("評価値の最大値:", rating_max)
print("観測された評価値:", observed_rating_values)
print("許可値に含まれない評価の件数:", len(invalid_ratings))

# 不正値が見つかった場合だけ、先頭10行を表示する
if not invalid_ratings.empty:
    display(invalid_ratings.head(10))

評価値の最小値: 0.5
評価値の最大値: 5.0
観測された評価値: [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
許可値に含まれない評価の件数: 0


### 5.4〜5.5の確認結果

- `ratings`に全列が同じ完全重複はなかった。
- 同じ`userId`と`movieId`を持つ評価行はなく、各ユーザー・映画の組み合わせに評価は1件だった。
- `movies`に完全重複はなく、識別キーである`movieId`の重複もなかった。
- `rating`の最小値は0.5、最大値は5.0だった。
- 観測された評価値は0.5〜5.0の0.5刻みで、許可値に含まれない評価は0件だった。
- 重複による二重集計や範囲外評価の問題は見つからなかったため、現時点で削除・補正は行わない。